# GSE44076 Series Matrix: Initial Loading Check

This notebook performs a first-pass structural inspection of the local GEO series matrix. It verifies that the file can be read, previews available sample metadata, locates the expression table, and checks the loaded matrix at a high level.

This stage does not perform normalization, dimensionality reduction, clustering, differential expression, biomarker selection, or biological interpretation.

## Setup

Paths are resolved relative to the repository root so the notebook can be run from either the project root or the `notebooks/` directory.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.geo_loader import (  # noqa: E402
    extract_geo_metadata,
    load_geo_expression_table,
    read_geo_series_lines,
)

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "GSE44076_series_matrix.txt.gz"
pd.set_option("display.max_columns", 10)

## File Check

The raw GEO file is intentionally excluded from Git. If it is absent, the message below identifies the expected local path.

In [2]:
if not RAW_PATH.is_file():
    raise FileNotFoundError(
        "GSE44076 series matrix not found. Download it from GEO and place it at "
        f"{RAW_PATH}"
    )

print(f"Found raw file: {RAW_PATH.relative_to(PROJECT_ROOT)}")
print(f"Compressed size: {RAW_PATH.stat().st_size / 1_000_000:.1f} MB")

Found raw file: data\raw\GSE44076_series_matrix.txt.gz
Compressed size: 32.6 MB


## GEO Metadata

GEO metadata records appear before the expression table and begin with `!`. The parser retains repeated values so sample-level fields can be inspected without changing their content.

In [3]:
series_lines = read_geo_series_lines(RAW_PATH)
metadata_lines = [line.rstrip() for line in series_lines if line.startswith("!")]
metadata = extract_geo_metadata(series_lines)

print(f"Total text lines: {len(series_lines):,}")
print(f"Metadata lines: {len(metadata_lines):,}")
display(pd.DataFrame({"metadata_line": metadata_lines[:10]}))

Total text lines: 49,469
Metadata lines: 81


,metadata_line
0,"!Series_title\t""Gene expression data from heal..."
1,"!Series_geo_accession\t""GSE44076"""
2,"!Series_status\t""Public on Mar 14 2014"""
3,"!Series_submission_date\t""Feb 05 2013"""
4,"!Series_last_update_date\t""Aug 31 2023"""
5,"!Series_pubmed_id\t""25215506"""
6,"!Series_pubmed_id\t""25253512"""
7,"!Series_pubmed_id\t""24597571"""
8,"!Series_pubmed_id\t""24760461"""
9,"!Series_pubmed_id\t""24839936"""


In [4]:
sample_accessions = metadata.get("Sample_geo_accession", [])
sample_titles = metadata.get("Sample_title", [])
source_names = metadata.get("Sample_source_name_ch1", [])
characteristics = metadata.get("Sample_characteristics_ch1", [])

sample_preview = pd.DataFrame(
    {
        "accession": sample_accessions,
        "title": sample_titles,
        "source_name": source_names,
    }
)

print(f"Sample accessions identified: {len(sample_accessions)}")
display(sample_preview.head())
print(f"Sample characteristic entries: {len(characteristics)}")
display(pd.Series(characteristics[:10], name="characteristic_preview").to_frame())

Sample accessions identified: 246


,accession,title,source_name
0,GSM1077598,Mucosa sample from A2119 healthy donnor,Healthy colon mucosa cells
1,GSM1077599,Mucosa sample from A2142 healthy donnor,Healthy colon mucosa cells
2,GSM1077600,Mucosa sample from B2104 healthy donnor,Healthy colon mucosa cells
3,GSM1077601,Mucosa sample from B2127 healthy donnor,Healthy colon mucosa cells
4,GSM1077602,Mucosa sample from B2150 healthy donnor,Healthy colon mucosa cells


Sample characteristic entries: 1476


,characteristic_preview
0,sample type: Mucosa
1,sample type: Mucosa
2,sample type: Mucosa
3,sample type: Mucosa
4,sample type: Mucosa
5,sample type: Mucosa
6,sample type: Mucosa
7,sample type: Mucosa
8,sample type: Mucosa
9,sample type: Mucosa


## Expression Table Location

The expression matrix is enclosed by GEO's `!series_matrix_table_begin` and `!series_matrix_table_end` markers.

In [5]:
table_markers = {
    line.strip(): line_number
    for line_number, line in enumerate(series_lines, start=1)
    if line.strip().startswith("!series_matrix_table_")
}
display(pd.Series(table_markers, name="line_number").to_frame())

,line_number
!series_matrix_table_begin,81
!series_matrix_table_end,49469


## Load the Expression Matrix

The loader preserves the `ID_REF` probe identifier column and reads each GSM accession as a sample column. No transformation is applied.

In [6]:
expression = load_geo_expression_table(RAW_PATH)

print(f"Expression table shape: {expression.shape[0]:,} rows x {expression.shape[1]:,} columns")
display(expression.iloc[:5, :8])

Expression table shape: 49,386 rows x 247 columns


,ID_REF,GSM1077598,GSM1077599,GSM1077600,GSM1077601,GSM1077602,GSM1077603,GSM1077604
0,11715100_at,3.6599,2.5712,3.3174,3.1835,2.9949,3.4481,3.1692
1,11715101_s_at,4.1524,4.1879,4.1040,4.3635,4.5227,4.1203,4.2486
2,11715102_x_at,3.3896,3.1228,3.2434,3.1175,3.3627,2.8233,3.4594
3,11715103_x_at,3.8929,3.7299,4.1048,3.3426,3.3376,3.0590,3.2893
4,11715104_s_at,7.3799,7.7642,6.4705,7.0046,6.9029,6.1424,6.5051


## Data Types and Rough Missingness Check

These checks confirm the parsed structure only. A zero missing-value count does not establish that every value is analytically valid or that the matrix is ready for downstream inference.

In [7]:
dtype_summary = expression.dtypes.astype(str).value_counts().rename_axis("dtype").to_frame("column_count")
missing_by_column = expression.isna().sum()

display(dtype_summary)
print(f"Total missing values detected: {int(missing_by_column.sum()):,}")
print(f"Columns containing missing values: {int(missing_by_column.gt(0).sum()):,}")

,column_count
dtype,
float64,246
object,1


Total missing values detected: 0
Columns containing missing values: 0


## Scope of This Check

The local file is readable and contains sample metadata plus a rectangular expression table. Before any biological analysis, later work should verify platform annotation, sample-group definitions, preprocessing history, and whether additional quality-control or normalization steps are appropriate.